### Hypothesis Testing Notebook — V4
**Model structure:**
- Model 0: Baseline (controls + firm/year FE)
- Model 1: Linear main effects (H1/H2 linear)
- Model 2: Quadratic main effects (H1/H2 inverted-U)
- Model 3a: Linear × Spatial moderation (H3a - slope moderation)
- Model 3b: Quadratic × Spatial moderation (H3b - curvature moderation)

**Additional Tests:**
- M3a (linear interactions) and M3b (quadratic interactions) as parallel moderation specs
- Hausman-style FE vs RE comparison on best FE model

#### Setup & Load Data

In [ ]:
# pip install stargazer statsmodels scikit-learn openpyxl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Poisson, NegativeBinomial
from statsmodels.tools.tools import add_constant
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
df_raw = pd.read_csv('firm_year_master_panel.csv')
print(f'Raw data shape: {df_raw.shape}')
print(df_raw['specification'].value_counts())

#### Data Prep

In [ ]:
def f_prepare_scaled_data(df_raw, time_window):
    closure_var   = f'delta_density_lag{time_window}'
    brokerage_var = f'delta_avg_betweenness_lag{time_window}'
    spatial_var   = 'mean_inv_inv_km'
    endog_var     = 'n_patents'
    spec_label    = f'time_window_{time_window}_network_years'

    df_win = df_raw[df_raw['specification'] == spec_label].copy().reset_index(drop=True)
    print(f'\n Preparing data for {time_window}-year window')
    print(f'  Rows after window filter: {df_win.shape[0]}')

    df_win['n_inv_inv_obs_log']  = np.log1p(df_win['n_inv_inv_obs'].fillna(0))
    df_win['n_inv_firm_obs_log'] = np.log1p(df_win['n_inv_firm_obs'].fillna(0))
    controls = ['n_inv_inv_obs_log', 'n_inv_firm_obs_log']

    key_cols = [endog_var] + controls + [closure_var, brokerage_var, spatial_var]
    df_win   = df_win.dropna(subset=key_cols).reset_index(drop=True)
    print(f'  Rows after NA drop: {df_win.shape[0]}')

    assert (df_win[endog_var] >= 0).all()
    print(f'  Zero patent share: {(df_win[endog_var] == 0).mean()*100:.1f}%')

    scale_cols = [closure_var, brokerage_var, spatial_var]
    scaler     = StandardScaler()
    df_win[scale_cols] = scaler.fit_transform(df_win[scale_cols])
    print(f'  Scaler fitted on {scale_cols}')

    # Quadratic terms
    closure_sq   = f'{closure_var}_sq'
    brokerage_sq = f'{brokerage_var}_sq'
    df_win[closure_sq]   = df_win[closure_var] ** 2
    df_win[brokerage_sq] = df_win[brokerage_var] ** 2

    # Linear interaction terms (M3a)
    closure_interact   = f'{closure_var}_x_spatial'
    brokerage_interact = f'{brokerage_var}_x_spatial'
    df_win[closure_interact]   = df_win[closure_var]   * df_win[spatial_var]
    df_win[brokerage_interact] = df_win[brokerage_var] * df_win[spatial_var]

    # Quadratic interaction terms (M3b)
    closure_sq_interact   = f'{closure_sq}_x_spatial'
    brokerage_sq_interact = f'{brokerage_sq}_x_spatial'
    df_win[closure_sq_interact]   = df_win[closure_sq]   * df_win[spatial_var]
    df_win[brokerage_sq_interact] = df_win[brokerage_sq] * df_win[spatial_var]

    n_firms = df_win['gvkeyUO'].nunique()
    n_years = df_win['network_year'].nunique()
    firm_dummies = pd.get_dummies(df_win['gvkeyUO'],  prefix='firm', drop_first=True).astype(float)
    year_dummies = pd.get_dummies(df_win['network_year'], prefix='year', drop_first=True).astype(float)
    print(f'  Unique firms: {n_firms}  Unique years: {n_years}')

    return {
        'df': df_win, 'endog': endog_var, 'controls': controls,
        'closure': closure_var, 'brokerage': brokerage_var,
        'closure_sq': closure_sq, 'brokerage_sq': brokerage_sq,
        'spatial': spatial_var,
        'closure_interact':    closure_interact,
        'brokerage_interact':  brokerage_interact,
        'closuresqinteract':   closure_sq_interact,
        'brokeragesqinteract': brokerage_sq_interact,
        'firm_dummies': firm_dummies, 'year_dummies': year_dummies,
        'groups': df_win['gvkeyUO'],
        'scaler': scaler, 'n_firms': n_firms, 'n_years': n_years,
        'n_obs': len(df_win), 'time_window': time_window,
    }

data_4y = f_prepare_scaled_data(df_raw, 4)
data_3y = f_prepare_scaled_data(df_raw, 3)
data_5y = f_prepare_scaled_data(df_raw, 5)


#### VIF Checks

In [ ]:
def f_compute_vif_report(data_dict, model_stage='full'):
    d   = data_dict
    df  = d['df']
    tw  = d['time_window']

    baseline_vars = d['controls']
    linear_vars   = baseline_vars + [d['closure'], d['brokerage']]
    quad_vars     = linear_vars   + [d['closure_sq'], d['brokerage_sq']]
    m3a_vars      = quad_vars     + [d['spatial'], d['closure_interact'],   d['brokerage_interact']]
    m3b_vars      = quad_vars     + [d['spatial'], d['closuresqinteract'],  d['brokeragesqinteract']]

    stage_map = {
        'baseline': baseline_vars, 'linear': linear_vars,
        'quadratic': quad_vars,    'm3a': m3a_vars,  'm3b': m3b_vars,
    }
    if model_stage not in stage_map:
        raise ValueError(f'model_stage must be one of {list(stage_map.keys())}')

    vars_to_check = stage_map[model_stage]
    X  = df[vars_to_check].dropna()
    X  = sm.add_constant(X)
    vif_data = pd.DataFrame({
        'Variable': X.columns,
        'VIF':      [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }).set_index('Variable').drop('const')
    vif_data['Status'] = vif_data['VIF'].apply(
        lambda v: 'HIGH >10' if v > 10 else ('MODERATE 5-10' if v > 5 else 'OK <5'))

    print(f'\n VIF Report {tw}-year window | Stage: {model_stage}')
    print(vif_data.round(3).to_string())
    n_high = (vif_data['VIF'] > 10).sum()
    n_mod  = ((vif_data['VIF'] > 5) & (vif_data['VIF'] <= 10)).sum()
    if n_high > 0:
        print(f' WARNING: {n_high} variables exceed VIF threshold of 10.')
    elif n_mod > 0:
        print(f' NOTE: {n_mod} variables in moderate range (5-10). Monitor.')
    else:
        print(' All variables within acceptable VIF range (<5).')
    return vif_data

# Run all stages for 4-year window
for stage in ['baseline', 'linear', 'quadratic', 'm3a', 'm3b']:
    f_compute_vif_report(data_4y, model_stage=stage)

#### Model Fitting Functions

In [ ]:
def f_build_exog(df, varlist, firm_dummies, year_dummies):
    X = pd.concat([
        df[varlist].reset_index(drop=True),
        year_dummies.reset_index(drop=True),
        firm_dummies.reset_index(drop=True),
    ], axis=1)
    return sm.add_constant(X, has_constant='add')

def f_fit_fe_poisson(data_dict, varlist, model_label, maxiter=500):
    d      = data_dict
    endog  = d['df'][d['endog']].reset_index(drop=True)
    exog   = f_build_exog(d['df'], varlist, d['firm_dummies'], d['year_dummies'])
    groups = d['groups'].reset_index(drop=True)
    print(f'  Fitting FE Poisson {model_label} | N={len(endog)}, K={exog.shape[1]}')
    try:
        model  = Poisson(endog, exog)
        result = model.fit(cov_type='cluster', cov_kwds={'groups': groups},
                           maxiter=maxiter, disp=False, warn_convergence=True)
        if not result.mle_retvals['converged']:
            print(f'  WARNING: {model_label} did not converge.')
        else:
            print(f'  Converged. Log-likelihood: {result.llf:.2f}')
        return result
    except Exception as e:
        print(f'  ERROR fitting {model_label}: {e}')
        return None

def f_fit_pooled_nb(data_dict, varlist, model_label, maxiter=500):
    d      = data_dict
    endog  = d['df'][d['endog']].reset_index(drop=True)
    X = pd.concat([
        d['df'][varlist].reset_index(drop=True),
        d['year_dummies'].reset_index(drop=True),
    ], axis=1)
    X      = sm.add_constant(X, has_constant='add')
    groups = d['groups'].reset_index(drop=True)
    print(f'  Fitting Pooled NB {model_label} | N={len(endog)}, K={X.shape[1]}')
    try:
        model  = NegativeBinomial(endog, X)
        result = model.fit(cov_type='cluster', cov_kwds={'groups': groups},
                           maxiter=maxiter, disp=False, warn_convergence=True)
        if not result.mle_retvals['converged']:
            print(f'  WARNING: {model_label} did not converge.')
        else:
            print(f'  Converged. Log-likelihood: {result.llf:.2f}')
        return result
    except Exception as e:
        print(f'  ERROR fitting {model_label}: {e}')
        return None

def f_compute_turning_point(result, linear_var, quad_var):
    if result is None: return {'turning_point': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan, 'b1': np.nan, 'b2': np.nan, 'direction': 'NA'}
    params = result.params
    cov    = result.cov_params()
    if linear_var not in params.index or quad_var not in params.index:
        return {'turning_point': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan, 'b1': np.nan, 'b2': np.nan, 'direction': 'NA'}
    b1, b2 = params[linear_var], params[quad_var]
    if abs(b2) < 1e-10:
        return {'turning_point': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan, 'b1': b1, 'b2': b2, 'direction': 'linear b2≈0'}
    tp    = -b1 / (2 * b2)
    db1   =  1  / (2 * b2)
    db2   = -b1 / (2 * b2**2)
    var_tp = db1**2*cov.loc[linear_var,linear_var] + db2**2*cov.loc[quad_var,quad_var] + 2*db1*db2*cov.loc[linear_var,quad_var]
    se_tp  = np.sqrt(max(var_tp, 0))
    return {
        'turning_point': tp, 'ci_lower': tp - 1.96*se_tp, 'ci_upper': tp + 1.96*se_tp,
        'b1': b1, 'b2': b2,
        'direction': 'Inverted-U (b2<0)' if b2 < 0 else 'U-shaped (b2>0)',
    }

def f_extract_results(result, model_label, data_dict, substantive_vars,
                      closure_linear=None, closure_quad=None,
                      brokerage_linear=None, brokerage_quad=None):
    if result is None: return None
    d = data_dict
    params = result.params;  bse = result.bse
    tvals  = result.tvalues; pvals = result.pvalues
    ci95   = result.conf_int(alpha=0.05)
    ci90   = result.conf_int(alpha=0.10)
    n_obs  = int(result.nobs)
    llf    = result.llf;  aic = result.aic;  bic = result.bic
    ll_null   = result.llnull
    pseudo_r2 = 1 - (llf / ll_null) if ll_null != 0 else np.nan
    tp_closure   = f_compute_turning_point(result, closure_linear,   closure_quad)   if closure_linear   else None
    tp_brokerage = f_compute_turning_point(result, brokerage_linear, brokerage_quad) if brokerage_linear else None
    return {
        'label': model_label, 'params': params, 'bse': bse, 'tvals': tvals,
        'pvals': pvals, 'ci95': ci95, 'ci90': ci90,
        'n_obs': n_obs, 'n_firms': d['n_firms'], 'llf': llf,
        'aic': aic, 'bic': bic, 'pseudo_r2': pseudo_r2,
        'tp_closure': tp_closure, 'tp_brokerage': tp_brokerage,
        'substantive_vars': substantive_vars,
    }

#### Run Models (M0, M1, M2, M3a, M3b)

In [ ]:
def f_run_models(data_dict, estimator='fe_poisson'):
    d      = data_dict
    tw     = d['time_window']
    fit_fn = f_fit_fe_poisson if estimator == 'fe_poisson' else f_fit_pooled_nb
    tag    = 'FE-Poisson' if estimator == 'fe_poisson' else 'Pooled-NB'

    print(f'\n{"="*60}')
    print(f'  Running {tag} | {tw}-year window')
    print(f'{"="*60}')

    vars_m0  = d['controls']
    vars_m1  = d['controls'] + [d['closure'], d['brokerage']]
    vars_m2  = d['controls'] + [d['closure'], d['brokerage'], d['closure_sq'], d['brokerage_sq']]
    vars_m3a = d['controls'] + [d['closure'], d['brokerage'], d['closure_sq'], d['brokerage_sq'],
                                 d['spatial'], d['closure_interact'], d['brokerage_interact']]
    vars_m3b = d['controls'] + [d['closure'], d['brokerage'], d['closure_sq'], d['brokerage_sq'],
                                 d['spatial'], d['closuresqinteract'], d['brokeragesqinteract']]

    print('\n--- Model 0: Baseline ---')
    res_m0 = fit_fn(d, vars_m0,  f'M0-{tw}y-{tag}')
    print('\n--- Model 1: Linear Main Effects ---')
    res_m1 = fit_fn(d, vars_m1,  f'M1-{tw}y-{tag}')
    print('\n--- Model 2: Quadratic Main Effects ---')
    res_m2 = fit_fn(d, vars_m2,  f'M2-{tw}y-{tag}')
    print('\n--- Model 3a: Linear × Spatial (slope moderation) ---')
    res_m3a = fit_fn(d, vars_m3a, f'M3a-{tw}y-{tag}')
    print('\n--- Model 3b: Quadratic × Spatial (curvature moderation) ---')
    res_m3b = fit_fn(d, vars_m3b, f'M3b-{tw}y-{tag}')

    ext_m0  = f_extract_results(res_m0,  f'Model 0 {tw}y',  d, vars_m0)
    ext_m1  = f_extract_results(res_m1,  f'Model 1 {tw}y',  d, vars_m1)
    ext_m2  = f_extract_results(res_m2,  f'Model 2 {tw}y',  d, vars_m2,
                                 d['closure'], d['closure_sq'], d['brokerage'], d['brokerage_sq'])
    ext_m3a = f_extract_results(res_m3a, f'Model 3a {tw}y', d, vars_m3a,
                                 d['closure'], d['closure_sq'], d['brokerage'], d['brokerage_sq'])
    ext_m3b = f_extract_results(res_m3b, f'Model 3b {tw}y', d, vars_m3b,
                                 d['closure'], d['closure_sq'], d['brokerage'], d['brokerage_sq'])

    # Delta LL tests
    print(f'\n--- Log-Likelihood Difference Tests | {tw}y {tag} ---')
    # Nested pairs: sequential M0→M1→M2, then both M3a and M3b branch from M2
    nested_pairs = [
        (ext_m0,  ext_m1,  res_m0,  res_m1,  'M0 → M1'),
        (ext_m1,  ext_m2,  res_m1,  res_m2,  'M1 → M2'),
        (ext_m2,  ext_m3a, res_m2,  res_m3a, 'M2 → M3a  (M2 + linear×spatial)'),
        (ext_m2,  ext_m3b, res_m2,  res_m3b, 'M2 → M3b  (M2 + quad×spatial)'),
    ]

    for base_ext, full_ext, base_res, full_res, label in nested_pairs:
        if base_ext is None or full_ext is None:
            print(f'  {label}: skipped (convergence failure)')
            continue
        delta_ll = 2 * (full_ext['llf'] - base_ext['llf'])
        df_diff  = len(full_res.params) - len(base_res.params)
        p_val    = stats.chi2.sf(delta_ll, df_diff)
        stars    = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else '†' if p_val < 0.10 else 'n.s.'
        print(f'  {label}: ΔLL={delta_ll:.2f}, df={df_diff}, p={p_val:.4f} {stars}')

    # Non-nested: M3a vs M3b — AIC/BIC comparison only
    print(f'\n  M3a vs M3b (non-nested parallels) — AIC/BIC only:')
    if ext_m3a and ext_m3b:
        aic_winner = 'M3a' if res_m3a.aic < res_m3b.aic else 'M3b'
        bic_winner = 'M3a' if res_m3a.bic < res_m3b.bic else 'M3b'
        print(f'  AIC:  M3a={res_m3a.aic:.2f}  M3b={res_m3b.aic:.2f}  → {aic_winner} preferred')
        print(f'  BIC:  M3a={res_m3a.bic:.2f}  M3b={res_m3b.bic:.2f}  → {bic_winner} preferred')
        print(f'  NOTE: BIC penalises complexity more heavily — '
            f'prefer BIC winner for parsimony argument.')

    return {
        'M0': ext_m0, 'M1': ext_m1, 'M2': ext_m2,
        'M3a': ext_m3a, 'M3b': ext_m3b,
        'raw': {'M0': res_m0, 'M1': res_m1, 'M2': res_m2,
                'M3a': res_m3a, 'M3b': res_m3b},
    }

results_4y_fe = f_run_models(data_4y, estimator='fe_poisson')

In [ ]:
print(' Quick Coefficient Check — 4-year FE Poisson')
focus_vars = [
    data_4y['closure'], data_4y['brokerage'],
    data_4y['closure_sq'], data_4y['brokerage_sq'],
    data_4y['spatial'],
    data_4y['closure_interact'],   data_4y['brokerage_interact'],
    data_4y['closuresqinteract'],  data_4y['brokeragesqinteract'],
]
rows = []
for mkey in ['M1', 'M2', 'M3a', 'M3b']:
    ext = results_4y_fe[mkey]
    if ext is None: continue
    for v in focus_vars:
        if v in ext['params'].index:
            rows.append({'Model': mkey, 'Variable': v,
                         'Coef': round(ext['params'][v], 4),
                         'SE':   round(ext['bse'][v], 4),
                         'p':    round(ext['pvals'][v], 4)})
coef_df = pd.DataFrame(rows)
print(coef_df.to_string(index=False))

print('\n Turning Points — 4-year FE Poisson')
for mkey in ['M2', 'M3a', 'M3b']:
    ext = results_4y_fe[mkey]
    if ext is None: continue
    print(f'\n  {mkey}')
    for name, tp_key in [('Closure (H1)', 'tp_closure'), ('Brokerage (H2)', 'tp_brokerage')]:
        tp = ext[tp_key]
        if tp:
            print(f'    {name}: TP={tp["turning_point"]:.4f} '
                  f'[{tp["ci_lower"]:.4f}, {tp["ci_upper"]:.4f}]  {tp["direction"]}')

In [ ]:
# Robustness checks for 3-year and 5-year windows, and pooled NB for 4-year window
results_3y_fe = f_run_models(data_3y, estimator='fe_poisson')
results_5y_fe = f_run_models(data_5y, estimator='fe_poisson')
results_4y_nb = f_run_models(data_4y, estimator='pooled_nb')

#### FE vs RE Comparison

SECTION — FE vs RE Hausman-style comparison
Strategy:
- Mundlak (1978) auxiliary regression — adds firm-mean of each time-varying variable as a proxy for unobserved heterogeneity.
- If Mundlak means are jointly significant → FE is necessary (RE inconsistent).
- This is the correct test for Poisson QMLE (Wooldridge 2010, pp. 620-622).

A formal Hausman test is not directly applicable to Poisson QMLE because the RE Poisson estimator is not efficient under the null in the same way as in linear models. The Mundlak approach is the standard alternative.

In [ ]:
print('='*70)
print('  Hausman-style FE vs RE Test — Mundlak (1978) Auxiliary Regression')
print('  Best FE model: M3a (primary) and M3b (Quad interaction terms), 4-year')
print('='*70)

def f_mundlak_test(data_dict, varlist, model_label):
    """
    Mundlak test for Poisson panel models.
    Adds firm-means of all time-varying substantive variables.
    Joint significance of firm-mean terms → FE preferred over RE.
    """
    d   = data_dict
    df  = d['df'].copy()
    tw  = d['time_window']

    # Time-varying vars only (exclude controls which are already logs of counts)
    time_varying = [v for v in varlist if v not in d['controls']]

    # Compute firm-means for each time-varying variable
    mundlak_cols = []
    for v in time_varying:
        col_name = f'{v}_firmmean'
        df[col_name] = df.groupby('gvkeyUO')[v].transform('mean')
        mundlak_cols.append(col_name)

    full_varlist = varlist + mundlak_cols
    endog  = df[d['endog']].reset_index(drop=True)

    # RE Poisson: year dummies only, no firm dummies, add Mundlak means
    X = pd.concat([
        df[full_varlist].reset_index(drop=True),
        d['year_dummies'].reset_index(drop=True),
    ], axis=1)
    X      = sm.add_constant(X, has_constant='add')
    groups = d['groups'].reset_index(drop=True)

    print(f'\n  Fitting Mundlak-augmented RE Poisson | {model_label} | N={len(endog)}, K={X.shape[1]}')
    try:
        model  = Poisson(endog, X)
        result = model.fit(cov_type='cluster', cov_kwds={'groups': groups},
                           maxiter=500, disp=False, warn_convergence=True)
        if not result.mle_retvals['converged']:
            print(f'  WARNING: did not converge.')
            return

        print(f'  Converged. Log-likelihood: {result.llf:.2f}')

        # Extract Mundlak mean coefficients and joint test
        params = result.params
        pvals  = result.pvalues
        bse    = result.bse

        print(f'\n  Mundlak firm-mean terms:')
        print(f'  {"Variable":<45} {"Coef":>8}  {"SE":>8}  {"p":>8}  Sig')
        any_sig = False
        for col in mundlak_cols:
            if col in params.index:
                p_v  = pvals[col]
                sig  = '***' if p_v < 0.001 else '**' if p_v < 0.01 else '*' if p_v < 0.05 else '†' if p_v < 0.10 else 'n.s.'
                if p_v < 0.10: any_sig = True
                print(f'  {col:<45} {params[col]:>8.4f}  {bse[col]:>8.4f}  {p_v:>8.4f}  {sig}')

        # Joint Wald test on Mundlak terms using restriction matrix
        from statsmodels.stats.contrast import ContrastResults
        mundlak_idx = [i for i, c in enumerate(result.params.index) if c in mundlak_cols]
        R = np.zeros((len(mundlak_idx), len(result.params)))
        for row, col_idx in enumerate(mundlak_idx):
            R[row, col_idx] = 1
        wald = result.wald_test(R, use_f=False)
        wald_stat = float(wald.statistic)
        wald_df   = int(wald.df_denom) if hasattr(wald, 'df_denom') else len(mundlak_idx)
        wald_p    = float(wald.pvalue)

        print(f'\n  Joint Wald test on all Mundlak means:')
        print(f'  χ²({len(mundlak_idx)}) = {wald_stat:.3f},  p = {wald_p:.4f}')
        if wald_p < 0.05:
            print(f'  => REJECT null (p={wald_p:.4f}): firm-means jointly significant.')
            print(f'     FE is necessary. RE Poisson is INCONSISTENT for this spec.')
            print(f'     FE Poisson QMLE confirmed as correct estimator.')
        else:
            print(f'  => FAIL TO REJECT null (p={wald_p:.4f}): firm-means not jointly significant.')
            print(f'     RE Poisson may be consistent, but FE is still robust.')

    except Exception as e:
        print(f'  ERROR: {e}')

# Run for M3a (primary) and M3b (quad interaction)
d   = data_4y
vars_m3a = d['controls'] + [d['closure'], d['brokerage'], d['closure_sq'], d['brokerage_sq'],
                              d['spatial'], d['closure_interact'], d['brokerage_interact']]
vars_m3b = d['controls'] + [d['closure'], d['brokerage'], d['closure_sq'], d['brokerage_sq'],
                              d['spatial'], d['closuresqinteract'], d['brokeragesqinteract']]

f_mundlak_test(data_4y, vars_m3a, 'M3a Linear×Spatial 4y')
f_mundlak_test(data_4y, vars_m3b, 'M3b Quad×Spatial 4y')

print('\n  Reference: Mundlak (1978), Econometrica 46(1), 69-85.')
print('  Reference: Wooldridge (2010), Econometric Analysis of Cross Section')
print('             and Panel Data, 2nd ed., pp. 620-622.')


#### Lind-Mehlum Test for Inv-U

- M2:  quadratic main effects only — formal test of H1/H2
- M3a: linear interactions — standard LM valid, no collinearity
- M3b: quadratic interactions — standard LM valid (b1/b2 not collinear with quad×spatial because quad×spatial is a separate moderator column)

In [ ]:
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '†'
    return 'n.s.'

def lind_mehlum_test(result, data_dict, linear_var, quad_var,
                     hypothesis='inverted-U'):
    df     = data_dict['df']
    params = result.params
    cov    = result.cov_params()
    b1     = params[linear_var]
    b2     = params[quad_var]

    x_obs  = df[linear_var]
    x_low  = x_obs.quantile(0.05)
    x_high = x_obs.quantile(0.95)

    if abs(b2) < 1e-10:
        return {'turning_point': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan,
                'b1': b1, 'b2': b2, 'direction': 'linear b2≈0',
                'x_low': x_low, 'x_high': x_high,
                'slope_low': np.nan, 'slope_high': np.nan,
                't_low': np.nan, 't_high': np.nan,
                'p_low': np.nan, 'p_high': np.nan,
                'p_joint': np.nan, 'tp_in_range': False, 'supported': False}

    tp    = -b1 / (2 * b2)
    db1   =  1  / (2 * b2)
    db2   = -b1 / (2 * b2**2)
    vb1   = cov.loc[linear_var, linear_var]
    vb2   = cov.loc[quad_var,   quad_var]
    cb12  = cov.loc[linear_var, quad_var]
    var_tp = db1**2 * vb1 + db2**2 * vb2 + 2 * db1 * db2 * cb12
    se_tp  = np.sqrt(max(var_tp, 0))
    ci_lower  = tp - 1.96 * se_tp
    ci_upper  = tp + 1.96 * se_tp
    direction = 'Inverted-U (b2<0)' if b2 < 0 else 'U-shaped (b2>0)'

    def slope_se(x_val):
        g   = np.array([1, 2 * x_val])
        idx = [linear_var, quad_var]
        V   = cov.loc[idx, idx].values
        return np.sqrt(max(float(g @ V @ g), 0))

    sl_lo = b1 + 2 * b2 * x_low
    sl_hi = b1 + 2 * b2 * x_high
    se_lo = slope_se(x_low);   se_hi = slope_se(x_high)
    t_lo  = sl_lo / se_lo if se_lo > 1e-8 else np.nan
    t_hi  = sl_hi / se_hi if se_hi > 1e-8 else np.nan

    if hypothesis == 'inverted-U':
        p_lo = stats.norm.sf( t_lo) if not np.isnan(t_lo) else np.nan
        p_hi = stats.norm.sf(-t_hi) if not np.isnan(t_hi) else np.nan
    else:
        p_lo = stats.norm.sf(-t_lo) if not np.isnan(t_lo) else np.nan
        p_hi = stats.norm.sf( t_hi) if not np.isnan(t_hi) else np.nan

    p_jt  = max(p_lo, p_hi) if not (np.isnan(p_lo) or np.isnan(p_hi)) else np.nan
    tp_in = (x_low < tp < x_high)
    supp  = (p_jt < 0.05) and tp_in if not np.isnan(p_jt) else False

    return {
        'turning_point': tp, 'ci_lower': ci_lower, 'ci_upper': ci_upper,
        'b1': b1, 'b2': b2, 'direction': direction,
        'x_low': x_low, 'x_high': x_high,
        'slope_low': sl_lo, 'slope_high': sl_hi,
        't_low': t_lo, 't_high': t_hi,
        'p_low': p_lo, 'p_high': p_hi,
        'p_joint': p_jt, 'tp_in_range': tp_in, 'supported': supp,
    }

def print_lm_report(lm, label, model_label):
    if lm is None: return
    print(f"\n  --- Lind-Mehlum Test | {label} | {model_label} ---")
    print(f"  b1 (linear):        {lm['b1']:.4f}")
    print(f"  b2 (quadratic):     {lm['b2']:.4f}")
    print(f"  Direction:          {lm['direction']}")
    print(f"  Test range:         [{lm['x_low']:.3f}, {lm['x_high']:.3f}] (5th–95th pct, SD units)")
    print(f"  Turning point:      {lm['turning_point']:.4f} SD units")
    print(f"  95% CI:             [{lm['ci_lower']:.4f}, {lm['ci_upper']:.4f}]")
    print(f"  TP within range:    {lm['tp_in_range']}")
    print(f"  Slope at x_low:     {lm['slope_low']:.4f}  (t={lm['t_low']:.3f}, p={lm['p_low']:.4f})")
    print(f"  Slope at x_high:    {lm['slope_high']:.4f} (t={lm['t_high']:.3f}, p={lm['p_high']:.4f})")
    print(f"  Joint p-value:      {lm['p_joint']:.4f} {sig_stars(lm['p_joint'])}")
    print(f"  Supported (p<0.05 AND TP in range): {lm['supported']}")

# ── Model 2 ───────────────────────────────────────────────────────────────────
print("=== Lind-Mehlum | MODEL 2 | 4-year FE Poisson ===")
res_m2 = results_4y_fe['raw']['M2']
for hyp_label, lv, qv, hyp in [
    ('H1 Closure (inverted-U)',       data_4y['closure'],   data_4y['closure_sq'],   'inverted-U'),
    ('H2 Brokerage (inverted-U)',     data_4y['brokerage'], data_4y['brokerage_sq'], 'inverted-U'),
    ('H2 Brokerage (U, exploratory)', data_4y['brokerage'], data_4y['brokerage_sq'], 'U-shape'),
]:
    lm = lind_mehlum_test(res_m2, data_4y, lv, qv, hypothesis=hyp)
    print_lm_report(lm, hyp_label, 'Model 2 (4y)')

# ── Model 3a: Linear interactions — standard LM valid ────────────────────────
print("\n=== Lind-Mehlum | MODEL 3a (Linear×Spatial) | 4-year FE Poisson ===")
res_m3a = results_4y_fe['raw']['M3a']
for hyp_label, lv, qv, hyp in [
    ('H1 Closure (inverted-U)',   data_4y['closure'],   data_4y['closure_sq'],   'inverted-U'),
    ('H2 Brokerage (inverted-U)', data_4y['brokerage'], data_4y['brokerage_sq'], 'inverted-U'),
]:
    lm = lind_mehlum_test(res_m3a, data_4y, lv, qv, hypothesis=hyp)
    print_lm_report(lm, hyp_label, 'Model 3a (4y)')

# ── Model 3b: Quadratic interactions — standard LM valid ─────────────────────
print("\n=== Lind-Mehlum | MODEL 3b (Quad×Spatial) | 4-year FE Poisson ===")
res_m3b = results_4y_fe['raw']['M3b']
for hyp_label, lv, qv, hyp in [
    ('H1 Closure (inverted-U)',   data_4y['closure'],   data_4y['closure_sq'],   'inverted-U'),
    ('H2 Brokerage (inverted-U)', data_4y['brokerage'], data_4y['brokerage_sq'], 'inverted-U'),
]:
    lm = lind_mehlum_test(res_m3b, data_4y, lv, qv, hypothesis=hyp)
    print_lm_report(lm, hyp_label, 'Model 3b (4y)')

# ── Turning point summary across all three models ─────────────────────────────
print("\n=== Turning Point Summary | 4-year FE Poisson ===")
for mkey, mlabel in [('M2', 'Model 2'), ('M3a', 'Model 3a'), ('M3b', 'Model 3b')]:
    res = results_4y_fe['raw'][mkey]
    print(f"\n  {mlabel}")
    for name, lv, qv in [
        ('Closure  (H1)', data_4y['closure'],   data_4y['closure_sq']),
        ('Brokerage (H2)', data_4y['brokerage'], data_4y['brokerage_sq']),
    ]:
        lm = lind_mehlum_test(res, data_4y, lv, qv)
        print(f"    {name}: TP={lm['turning_point']:.4f} "
              f"[{lm['ci_lower']:.4f}, {lm['ci_upper']:.4f}]  "
              f"In range [{lm['x_low']:.3f},{lm['x_high']:.3f}]: {lm['tp_in_range']}  "
              f"{lm['direction']}")


#### Visuals

##### Interaction Plots
- M3a: two-line plot showing slope moderation (linear×spatial)
- M3b: two-line plot showing curvature moderation (quad×spatial)

Both use mean ± 1SD of spatial moderator (Aiken & West, 1991)

In [ ]:
def f_plot_interaction(x_range, b_focal, b_focal_sq,
                       b_interact_lin, b_interact_sq,
                       b_mod, b_ctrl1, b_ctrl2, const,
                       mean_c1, mean_c2,
                       spatial_col_data,
                       focal_label, filename,
                       window_label='4Y window'):
    """
    Two-line moderation plot at mean ± 1SD of spatial moderator.
    Pass b_interact_lin=0.0 for M3b; pass b_interact_sq=0.0 for M3a.
    Aiken & West (1991).
    """
    z_mean = float(spatial_col_data.mean())
    z_sd   = float(spatial_col_data.std())
    z_low  = z_mean - z_sd
    z_high = z_mean + z_sd

    print(f"  Spatial: mean={z_mean:.3f}  SD={z_sd:.3f}  "
          f"low={z_low:.3f}  high={z_high:.3f}")

    scenarios = [
        (z_low,  'Low spatial dispersion (mean−1SD)',  '#16A34A', '-'),
        (z_high, 'High spatial dispersion (mean+1SD)', '#DC2626', '--'),
    ]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for z_val, label, color, ls in scenarios:
        log_mu = (const
                  + b_focal        * x_range
                  + b_focal_sq     * x_range**2
                  + b_interact_lin * x_range    * z_val
                  + b_interact_sq  * x_range**2 * z_val
                  + b_mod          * z_val
                  + b_ctrl1        * mean_c1
                  + b_ctrl2        * mean_c2)
        ax.plot(x_range, np.exp(log_mu),
                color=color, linestyle=ls, linewidth=2.5, label=label)

    ax.set_xlabel(focal_label,              fontsize=12)
    ax.set_ylabel('Predicted Patent Count', fontsize=12)
    ax.set_title(
        f'H3 — {focal_label} × Spatial Dispersion\n'
        f'{window_label}, other controls at mean',
        fontsize=12)
    ax.legend(fontsize=10, framealpha=0.9, loc='best')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {filename}')


df4            = data_4y['df']
mean_inv_pairs = df4['n_inv_inv_obs_log'].mean()
mean_inv_firm  = df4['n_inv_firm_obs_log'].mean()
x_clos = np.linspace(df4[data_4y['closure']].min(),   df4[data_4y['closure']].max(),   300)
x_brok = np.linspace(df4[data_4y['brokerage']].min(), df4[data_4y['brokerage']].max(), 300)

# ── M3a plots: b_interact_sq = 0.0 (no quad interaction) ─────────────────────
print("\n--- M3a Interaction Plots (Linear × Spatial — slope moderation) ---")
p_m3a = results_4y_fe['raw']['M3a'].params

f_plot_interaction(
    x_range          = x_clos,
    b_focal          = p_m3a[data_4y['closure']],
    b_focal_sq       = p_m3a[data_4y['closure_sq']],
    b_interact_lin   = p_m3a[data_4y['closure_interact']],
    b_interact_sq    = 0.0,
    b_mod            = p_m3a[data_4y['spatial']],
    b_ctrl1          = p_m3a['n_inv_inv_obs_log'],
    b_ctrl2          = p_m3a['n_inv_firm_obs_log'],
    const            = p_m3a.get('const', 0),
    mean_c1          = mean_inv_pairs,
    mean_c2          = mean_inv_firm,
    spatial_col_data = df4[data_4y['spatial']],
    focal_label      = 'Delta Closure',
    filename         = 'fig_m3a_closure_x_spatial.png',
    window_label     = '4Y window — M3a (Linear×Spatial)',
)

f_plot_interaction(
    x_range          = x_brok,
    b_focal          = p_m3a[data_4y['brokerage']],
    b_focal_sq       = p_m3a[data_4y['brokerage_sq']],
    b_interact_lin   = p_m3a[data_4y['brokerage_interact']],
    b_interact_sq    = 0.0,
    b_mod            = p_m3a[data_4y['spatial']],
    b_ctrl1          = p_m3a['n_inv_inv_obs_log'],
    b_ctrl2          = p_m3a['n_inv_firm_obs_log'],
    const            = p_m3a.get('const', 0),
    mean_c1          = mean_inv_pairs,
    mean_c2          = mean_inv_firm,
    spatial_col_data = df4[data_4y['spatial']],
    focal_label      = 'Delta Brokerage',
    filename         = 'fig_m3a_brokerage_x_spatial.png',
    window_label     = '4Y window — M3a (Linear×Spatial)',
)

# ── M3b plots: b_interact_lin = 0.0 (no linear interaction) ──────────────────
print("\n--- M3b Interaction Plots (Quadratic × Spatial — curvature moderation) ---")
p_m3b = results_4y_fe['raw']['M3b'].params

f_plot_interaction(
    x_range          = x_clos,
    b_focal          = p_m3b[data_4y['closure']],
    b_focal_sq       = p_m3b[data_4y['closure_sq']],
    b_interact_lin   = 0.0,
    b_interact_sq    = p_m3b[data_4y['closuresqinteract']],
    b_mod            = p_m3b[data_4y['spatial']],
    b_ctrl1          = p_m3b['n_inv_inv_obs_log'],
    b_ctrl2          = p_m3b['n_inv_firm_obs_log'],
    const            = p_m3b.get('const', 0),
    mean_c1          = mean_inv_pairs,
    mean_c2          = mean_inv_firm,
    spatial_col_data = df4[data_4y['spatial']],
    focal_label      = 'Delta Closure',
    filename         = 'fig_m3b_closure_x_spatial.png',
    window_label     = '4Y window — M3b (Quad×Spatial)',
)

f_plot_interaction(
    x_range          = x_brok,
    b_focal          = p_m3b[data_4y['brokerage']],
    b_focal_sq       = p_m3b[data_4y['brokerage_sq']],
    b_interact_lin   = 0.0,
    b_interact_sq    = p_m3b[data_4y['brokeragesqinteract']],
    b_mod            = p_m3b[data_4y['spatial']],
    b_ctrl1          = p_m3b['n_inv_inv_obs_log'],
    b_ctrl2          = p_m3b['n_inv_firm_obs_log'],
    const            = p_m3b.get('const', 0),
    mean_c1          = mean_inv_pairs,
    mean_c2          = mean_inv_firm,
    spatial_col_data = df4[data_4y['spatial']],
    focal_label      = 'Delta Brokerage',
    filename         = 'fig_m3b_brokerage_x_spatial.png',
    window_label     = '4Y window — M3b (Quad×Spatial)',
)


##### Marginal Effects Plots
- Shows the inverted-U / U-shape for Closure and Brokerage
- Plotted from M2 (clean main effects) and M3a (primary moderation model)
- All other vars held at observed means; spatial held at its mean (= 0, scaled)

In [ ]:
def f_plot_marginal(x_range, b_focal, b_focal_sq,
                    b_other_params,   # dict of {varname: coef} for controls + spatial
                    mean_other,       # dict of {varname: mean_value}
                    const,
                    focal_label, model_label, filename):
    """
    Single-curve marginal effect plot with turning point marker.
    b_other_params: all non-focal coefficients held at their means.
    """
    # Base prediction from controls + spatial (all held at mean)
    base = const + sum(b_other_params.get(v, 0) * mean_other.get(v, 0)
                       for v in b_other_params)

    log_mu = base + b_focal * x_range + b_focal_sq * x_range**2
    y_pred = np.exp(log_mu)

    # Turning point
    tp = -b_focal / (2 * b_focal_sq) if abs(b_focal_sq) > 1e-10 else np.nan

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(x_range, y_pred, color='#2563EB', linewidth=2.5, label='Predicted patent count')

    if not np.isnan(tp) and (x_range.min() <= tp <= x_range.max()):
        tp_y = np.exp(base + b_focal * tp + b_focal_sq * tp**2)
        ax.axvline(x=tp, color='#DC2626', linestyle='--', linewidth=1.5, alpha=0.8)
        ax.scatter([tp], [tp_y], color='#DC2626', s=80, zorder=5,
                   label=f'Turning point {tp:.3f}')

    ax.set_xlabel(focal_label,              fontsize=12)
    ax.set_ylabel('Predicted Patent Count', fontsize=12)
    ax.set_title(
        f'Marginal Effect — {focal_label} → Innovation Output\n'
        f'{model_label}, controls held at mean',
        fontsize=12)
    ax.legend(fontsize=10, framealpha=0.9, loc='best')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {filename}')


def run_marginal_plots(results_dict, data_dict, model_key, model_label):
    d      = data_dict
    df4    = d['df']
    res    = results_dict['raw'][model_key]
    p      = res.params

    # Control means
    mean_vals = {
        'n_inv_inv_obs_log': df4['n_inv_inv_obs_log'].mean(),
        'n_inv_firm_obs_log': df4['n_inv_firm_obs_log'].mean(),
        d['spatial']: 0.0,   # standardised → mean = 0
    }

    # Include spatial interact terms at mean spatial = 0
    # (their contribution vanishes at mean, which is correct)
    for extra in [d['closure_interact'], d['brokerage_interact'],
                  d['closuresqinteract'], d['brokeragesqinteract']]:
        mean_vals[extra] = 0.0

    # Build dict of non-focal coefficients present in this model
    def get_other_params(focal_vars):
        return {v: float(p[v]) for v in p.index
                if v not in focal_vars and v != 'const' and v in mean_vals}

    const = float(p.get('const', 0))

    x_clos = np.linspace(df4[d['closure']].min(),   df4[d['closure']].max(),   300)
    x_brok = np.linspace(df4[d['brokerage']].min(), df4[d['brokerage']].max(), 300)

    print(f'\n--- Marginal Effect Plots | {model_label} ---')

    # Closure
    f_plot_marginal(
        x_range       = x_clos,
        b_focal       = float(p[d['closure']]),
        b_focal_sq    = float(p[d['closure_sq']]),
        b_other_params= get_other_params([d['closure'], d['closure_sq']]),
        mean_other    = mean_vals,
        const         = const,
        focal_label   = 'Delta Closure',
        model_label   = model_label,
        filename      = f'fig_marginal_closure_{model_key.lower()}.png',
    )

    # Brokerage
    f_plot_marginal(
        x_range       = x_brok,
        b_focal       = float(p[d['brokerage']]),
        b_focal_sq    = float(p[d['brokerage_sq']]),
        b_other_params= get_other_params([d['brokerage'], d['brokerage_sq']]),
        mean_other    = mean_vals,
        const         = const,
        focal_label   = 'Delta Brokerage',
        model_label   = model_label,
        filename      = f'fig_marginal_brokerage_{model_key.lower()}.png',
    )


# ── Run for M2, M3a, M3b ─────────────────────────────────────────────────────
for mkey, mlabel in [
    ('M2',  'Model 2 — Quadratic Main Effects | 4Y window'),
    ('M3a', 'Model 3a — Linear×Spatial | 4Y window'),
    ('M3b', 'Model 3b — Quad×Spatial | 4Y window'),
]:
    run_marginal_plots(results_4y_fe, data_4y, mkey, mlabel)


#### Export results

One .xlsx per time window (3y, 4y, 5y), each with 4 sheets:
  - Sheet 1 — PrimaryResults  : M0, M1, M2, M3a, M3b log-coefs + fit stats
  - Sheet 2 — IRRModel3       : IRRs with 95% CI for M3a and M3b
  - Sheet 3 — RobustnessWindows : M3a/M3b coefs across 3y, 4y, 5y
  - Sheet 4 — RobustnessEstimator: FE Poisson vs Pooled NB (4y M3a/M3b only)

In [ ]:
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '†'
    return ''

def format_coef_cell(coef, se, p, decimals=4):
    stars = sig_stars(p)
    return (f'{coef:.{decimals}f}{stars}',
            f'({se:.{decimals}f})')

# ── Shared variable display map ───────────────────────────────────────────────
def get_var_display(d):
    return {
        d['controls'][0]:        'ln(Inventor-pair obs)',
        d['controls'][1]:        'ln(Inventors per firm)',
        d['closure']:            'Δ Closure (linear)',
        d['brokerage']:          'Δ Brokerage (linear)',
        d['closure_sq']:         'Δ Closure (squared)',
        d['brokerage_sq']:       'Δ Brokerage (squared)',
        d['spatial']:            'Spatial Dispersion (mean inv-inv km)',
        d['closure_interact']:   'Δ Closure × Spatial',
        d['brokerage_interact']: 'Δ Brokerage × Spatial',
        d['closuresqinteract']:  'Δ Closure² × Spatial',
        d['brokeragesqinteract']:'Δ Brokerage² × Spatial',
    }

def get_display_order(d):
    return [
        d['controls'][0],        d['controls'][1],
        d['closure'],            d['brokerage'],
        d['closure_sq'],         d['brokerage_sq'],
        d['spatial'],
        d['closure_interact'],   d['brokerage_interact'],
        d['closuresqinteract'],  d['brokeragesqinteract'],
    ]

# ── Sheet 1: Primary Results ──────────────────────────────────────────────────
def build_primary_table(results_dict, data_dict):
    d          = data_dict
    tw         = d['time_window']
    var_display = get_var_display(d)
    disp_order  = get_display_order(d)

    models  = ['M0', 'M1', 'M2', 'M3a', 'M3b']
    labels  = {
        'M0':  f'Model 0\nBaseline',
        'M1':  f'Model 1\nLinear Effects',
        'M2':  f'Model 2\nQuadratic Effects',
        'M3a': f'Model 3a\nLinear×Spatial',
        'M3b': f'Model 3b\nQuad×Spatial',
    }
    rows = []

    # Coefficient block
    for var, display_name in var_display.items():
        if var not in disp_order: continue
        coef_row = {'Variable': display_name, 'Type': 'coef'}
        se_row   = {'Variable': '',           'Type': 'se'}
        for mkey in models:
            ext  = results_dict.get(mkey)
            # map lag4 → lagN for robustness windows
            vartw = var.replace('lag4', f'lag{tw}')
            if ext and vartw in ext['params'].index:
                cs, ss = format_coef_cell(
                    ext['params'][vartw], ext['bse'][vartw], ext['pvals'][vartw])
                coef_row[mkey] = cs
                se_row[mkey]   = ss
            else:
                coef_row[mkey] = '—'
                se_row[mkey]   = ''
        rows.append(coef_row)
        rows.append(se_row)

    # Separator
    rows.append({'Variable': '─'*40, 'Type': 'sep',
                 **{m: '' for m in models}})

    # Turning point block
    tp_items = [
        ('Closure TP (SD units)',    'tp_closure',   'turning_point'),
        ('  95% CI Lower',           'tp_closure',   'ci_lower'),
        ('  95% CI Upper',           'tp_closure',   'ci_upper'),
        ('  Shape',                  'tp_closure',   'direction'),
        ('Brokerage TP (SD units)',  'tp_brokerage', 'turning_point'),
        ('  95% CI Lower',           'tp_brokerage', 'ci_lower'),
        ('  95% CI Upper',           'tp_brokerage', 'ci_upper'),
        ('  Shape',                  'tp_brokerage', 'direction'),
    ]
    for label, tp_key, field in tp_items:
        row = {'Variable': label, 'Type': 'tp'}
        for mkey in models:
            ext = results_dict.get(mkey)
            if ext and ext.get(tp_key) and field in ext[tp_key]:
                val = ext[tp_key][field]
                row[mkey] = (f'{val:.4f}' if isinstance(val, float) else str(val))
            else:
                row[mkey] = '—'
        rows.append(row)

    rows.append({'Variable': '─'*40, 'Type': 'sep',
                 **{m: '' for m in models}})

    # Fit statistics block
    fit_items = [
        ('N (Observations)',      'n_obs',     False),
        ('N (Firms)',             'n_firms',   False),
        ('Log-Likelihood',        'llf',       True),
        ('Pseudo-R² (McFadden)',  'pseudo_r2', True),
        ('AIC',                   'aic',       True),
        ('BIC',                   'bic',       True),
    ]
    for label, field, is_float in fit_items:
        row = {'Variable': label, 'Type': 'fit'}
        for mkey in models:
            ext = results_dict.get(mkey)
            if ext and field in ext:
                val = ext[field]
                row[mkey] = (f'{val:.2f}' if is_float else str(int(val)))
            else:
                row[mkey] = '—'
        rows.append(row)

    # Delta LL row
    row_dll = {'Variable': 'ΔLog-Likelihood vs. baseline', 'Type': 'fit'}

    dll_baselines = {
        'M0':  None,   # no baseline
        'M1':  'M0',   # M1 vs M0
        'M2':  'M1',   # M2 vs M1
        'M3a': 'M2',   # M3a vs M2  ← both branch from M2
        'M3b': 'M2',   # M3b vs M2  ← both branch from M2
    }

    for mkey in models:
        baseline_key = dll_baselines[mkey]
        ext_full     = results_dict.get(mkey)
        ext_base     = results_dict.get(baseline_key) if baseline_key else None

        if ext_full is None or ext_base is None:
            row_dll[mkey] = '—'
            continue

        dll     = 2 * (ext_full['llf'] - ext_base['llf'])
        df_diff = len(ext_full['params']) - len(ext_base['params'])

        if mkey in ('M3a', 'M3b'):
            # Non-nested with each other, but each is nested under M2
            p_dll = stats.chi2.sf(dll, df_diff)
            row_dll[mkey] = f'{dll:.2f}{sig_stars(p_dll)} (vs M2)'
        else:
            p_dll = stats.chi2.sf(dll, df_diff) if df_diff > 0 else np.nan
            row_dll[mkey] = (f'{dll:.2f}{sig_stars(p_dll)}'
                            if not np.isnan(p_dll) else f'{dll:.2f}')

    rows.append(row_dll)

    # Add AIC/BIC winner note as a separate info row
    row_aic = {'Variable': 'AIC (M3a vs M3b winner)', 'Type': 'fit'}
    row_bic = {'Variable': 'BIC (M3a vs M3b winner)', 'Type': 'fit'}
    for mkey in models:
        ext = results_dict.get(mkey)
        if mkey in ('M3a', 'M3b') and ext:
            row_aic[mkey] = f'{ext["aic"]:.2f}'
            row_bic[mkey] = f'{ext["bic"]:.2f}'
        else:
            row_aic[mkey] = '—'
            row_bic[mkey] = '—'
    rows.append(row_aic)
    rows.append(row_bic)

    # FE flags
    for label, val in [('Year Fixed Effects', 'Yes'), ('Firm Fixed Effects', 'Yes')]:
        row = {'Variable': label, 'Type': 'fit'}
        for mkey in models: row[mkey] = val
        rows.append(row)

    df_table = pd.DataFrame(rows, columns=['Variable', 'Type'] + models)
    df_table.columns = ['Variable', 'Type'] + [labels[m] for m in models]
    df_table = df_table.drop(columns=['Type'])
    return df_table

# ── Sheet 2: IRR Table (M2, M3a, M3b) ────────────────────────────────────────
def build_irr_table(results_dict, data_dict):
    d           = data_dict
    tw          = d['time_window']
    var_display = get_var_display(d)
    disp_order  = get_display_order(d)

    irr_models = [
        ('M2',  'Model 2\n(Quadratic)'),
        ('M3a', 'Model 3a\n(Linear×Spatial)'),
        ('M3b', 'Model 3b\n(Quad×Spatial)'),
    ]

    rows = []
    for var, display_name in var_display.items():
        if var not in disp_order: continue
        row = {'Variable': display_name}
        vartw = var.replace('lag4', f'lag{tw}')
        for mkey, mlabel in irr_models:
            ext = results_dict.get(mkey)
            if ext and vartw in ext['params'].index:
                b     = ext['params'][vartw]
                se    = ext['bse'][vartw]
                p     = ext['pvals'][vartw]
                irr   = np.exp(b)
                ci_lo = np.exp(b - 1.96 * se)
                ci_hi = np.exp(b + 1.96 * se)
                row[f'{mkey} IRR']     = round(irr, 4)
                row[f'{mkey} 95% CI']  = f'[{ci_lo:.4f}, {ci_hi:.4f}]'
                row[f'{mkey} p-value'] = round(p, 4)
                row[f'{mkey} Sig']     = sig_stars(p)
            else:
                row[f'{mkey} IRR']     = '—'
                row[f'{mkey} 95% CI']  = '—'
                row[f'{mkey} p-value'] = '—'
                row[f'{mkey} Sig']     = ''
        rows.append(row)
    return pd.DataFrame(rows)

# ── Sheet 3: Robustness Windows (M2, M3a, M3b across 3y/4y/5y) ───────────────
def build_robustness_windows(results_3y, results_4y, results_5y,
                              data_3y, data_4y, data_5y):
    d4          = data_4y
    var_display = get_var_display(d4)
    disp_order  = get_display_order(d4)

    rob_models = ['M2', 'M3a', 'M3b']
    rows = []
    for var4y, display_name in var_display.items():
        if var4y not in disp_order: continue
        for mkey in rob_models:
            coef_row = {'Variable': display_name, 'Model': mkey, 'Stat': 'Coef'}
            se_row   = {'Variable': '',           'Model': mkey, 'Stat': 'SE'}
            for tw, results_dict in [(3, results_3y), (4, results_4y), (5, results_5y)]:
                ext   = results_dict.get(mkey)
                vartw = var4y.replace('lag4', f'lag{tw}')
                if ext and vartw in ext['params'].index:
                    cs, ss = format_coef_cell(
                        ext['params'][vartw], ext['bse'][vartw], ext['pvals'][vartw])
                    coef_row[f'{tw}Y Window'] = cs
                    se_row[f'{tw}Y Window']   = ss
                else:
                    coef_row[f'{tw}Y Window'] = '—'
                    se_row[f'{tw}Y Window']   = ''
            rows.append(coef_row)
            rows.append(se_row)
    return pd.DataFrame(rows)

# ── Sheet 4: Robustness Estimator (FE Poisson vs Pooled NB, M2/M3a/M3b, 4y) ──
def build_robustness_estimator(results_fe, results_nb, data_dict):
    d           = data_dict
    tw          = d['time_window']
    var_display = get_var_display(d)
    disp_order  = get_display_order(d)

    rob_models = ['M2', 'M3a', 'M3b']
    rows = []
    for var, display_name in var_display.items():
        if var not in disp_order: continue
        vartw = var.replace('lag4', f'lag{tw}')
        for mkey in rob_models:
            coef_row = {'Variable': display_name, 'Model': mkey, 'Stat': 'Coef'}
            se_row   = {'Variable': '',           'Model': mkey, 'Stat': 'SE'}
            for est_label, results_dict in [('FE-Poisson', results_fe),
                                            ('Pooled-NB',  results_nb)]:
                ext = results_dict.get(mkey)
                if ext and vartw in ext['params'].index:
                    cs, ss = format_coef_cell(
                        ext['params'][vartw], ext['bse'][vartw], ext['pvals'][vartw])
                    coef_row[f'{est_label} Coef'] = cs
                    se_row[f'{est_label} SE']     = ss
                else:
                    coef_row[f'{est_label} Coef'] = '—'
                    se_row[f'{est_label} SE']     = ''
            rows.append(coef_row)
            rows.append(se_row)
    return pd.DataFrame(rows)

# ── Master export: 3 files × 4 sheets ────────────────────────────────────────
def export_to_excel(primary_table, irr_table, robustness_windows,
                    robustness_estimator, filename):
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        primary_table.to_excel(       writer, sheet_name='PrimaryResults',      index=False)
        irr_table.to_excel(           writer, sheet_name='IRRModel3',           index=False)
        robustness_windows.to_excel(  writer, sheet_name='RobustnessWindows',   index=False)
        robustness_estimator.to_excel(writer, sheet_name='RobustnessEstimator', index=False)

        wb = writer.book
        header_font = Font(bold=True)
        header_fill = PatternFill('solid', fgColor='D9E1F2')

        for sheet in wb.worksheets:
            # Bold + fill header row
            for cell in sheet[1]:
                cell.font      = header_font
                cell.fill      = header_fill
                cell.alignment = Alignment(horizontal='center', wrap_text=True)
            # Freeze header
            sheet.freeze_panes = 'A2'
            # Auto-width columns
            for col in sheet.columns:
                max_len = max(
                    (len(str(cell.value)) for cell in col if cell.value),
                    default=10)
                sheet.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

    print(f'  Exported → {filename}')

# ── Execute ───────────────────────────────────────────────────────────────────
print('Building robustness tables...')
rob_windows   = build_robustness_windows(results_3y_fe, results_4y_fe, results_5y_fe,
                                          data_3y,       data_4y,       data_5y)
rob_estimator = build_robustness_estimator(results_4y_fe, results_4y_nb, data_4y)

print('\nExporting Excel files...')
for tw, results_fe, data_dict in [
    (3, results_3y_fe, data_3y),
    (4, results_4y_fe, data_4y),
    (5, results_5y_fe, data_5y),
]:
    primary   = build_primary_table(results_fe, data_dict)
    irr       = build_irr_table(results_fe, data_dict)
    filename  = f'regression_results_{tw}year_V4.xlsx'
    export_to_excel(primary, irr, rob_windows, rob_estimator, filename)

print('\nDone. Files produced:')